In [ ]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [ ]:
import numpy as np
from bayes_opt import BayesianOptimization

def _vector_to_action_plan(x, num_od, total_step, block_size, demand_max):
    n_blocks = int(np.ceil(total_step / block_size))

    if demand_max == 1:
        x_int = (x >= 0.5).astype(np.int32)
    else:
        x_int = np.clip(np.rint(x), 0, demand_max).astype(np.int32)

    action_plan = np.zeros((total_step, num_od), dtype=np.int32)
    k = 0
    for b in range(n_blocks):
        s = b * block_size
        e = min((b + 1) * block_size, total_step)
        for od in range(num_od):
            action_plan[s:e, od] = x_int[k]
            k += 1
    return action_plan

def _evaluate_episode(build_env_fn, worker_idx, total_step, action_plan, seed):
    env, _, _ = build_env_fn(worker_idx=worker_idx, total_step=total_step, seed=seed)
    total_reward = 0.0
    try:
        for t in range(total_step):
            _, reward, terminated, truncated, _ = env.step(action_plan[t])
            total_reward += float(reward)
            if terminated or truncated:
                break
    finally:
        env.close()
    return total_reward

def bayes_optimize(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    init_points=0,
    n_iter=10,
    seed=42,
    block_size=1,
    demand_max=1,
    verbose=2,
):
    if block_size < 1:
        raise ValueError("block_size must be >= 1")
    if demand_max < 1:
        raise ValueError("demand_max must be >= 1")

    n_blocks = int(np.ceil(total_step / block_size))
    dim = n_blocks * num_od
    key_order = [f"x{i}" for i in range(dim)]
    pbounds = {k: (0.0, float(demand_max)) for k in key_order}

    eval_count = {"n": 0}

    def objective(**kwargs):
        x = np.array([kwargs[k] for k in key_order], dtype=np.float32)
        action_plan = _vector_to_action_plan(
            x=x,
            num_od=num_od,
            total_step=total_step,
            block_size=block_size,
            demand_max=demand_max,
        )
        score = _evaluate_episode(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            total_step=total_step,
            action_plan=action_plan,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1
        return float(score)

    optimizer = BayesianOptimization(
        f=objective,
        pbounds=pbounds,
        random_state=seed,
        verbose=verbose,
        allow_duplicate_points=True,
    )
    optimizer.maximize(init_points=init_points, n_iter=n_iter)

    best_x = np.array([optimizer.max["params"][k] for k in key_order], dtype=np.float32)
    best_actions = _vector_to_action_plan(
        x=best_x,
        num_od=num_od,
        total_step=total_step,
        block_size=block_size,
        demand_max=demand_max,
    )
    best_value = float(optimizer.max["target"])
    return best_actions, best_value, optimizer

In [ ]:
import time
from trial_timing import reset_trial_times, append_trial_time
import json
from config import Config

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()

    block_size = 1
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, best_score, bo = bayes_optimize(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        init_points=0,
        n_iter=300,
        seed=seed_opt,
        block_size=block_size,
        demand_max=1,
        verbose=2,
    )

    print("Best total_reward:", best_score)
    print("Best action shape:", best_actions.shape)

    env, obs, info = build_env(worker_idx=0, seed=seed_eval)
    print("reset complete, obs shape:", obs.shape)

    for t in range(Config.TOTAL_STEP):
        print(f"step: {t+1}")
        action = best_actions[t]
        obs, reward, terminated, truncated, info = env.step(action)

    env.close()

    trajectory = info.get("trajectory", [])
    with open(Config.RESULT_DIR + f"/trajectory_{trial}.json", "w", encoding="utf-8") as f:
        json.dump(trajectory, f, ensure_ascii=False, indent=2)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)

In [ ]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
